# Stock Market Analytics and Portfolio Performance Dashboard

Analyzing: **Apple (AAPL), Microsoft (MSFT), TCS (TCS.NS), Infosys (INFY.NS), Reliance (RELIANCE.NS)**

This notebook covers:
1. Data collection
2. Returns
3. Volatility & moving averages
4. Risk metrics (Sharpe, Sortino, Calmar, VaR, CVaR, Max Drawdown, beta)
5. Portfolio comparison (correlation, equal-weight, efficient frontier, optimizer)
6. Bonus: forecasting (linear trend + ARIMA, with a backtest to check accuracy)

> Run `python src/data_collection.py` first to populate `data/processed/combined_prices.csv`.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from config import TICKERS, START_DATE
import analytics as an
import risk_metrics as rm
import portfolio as pf
import forecasting as fc

pd.options.display.float_format = '{:.4f}'.format

## 1. Load data

In [ ]:
combined = pd.read_csv('../data/processed/combined_prices.csv', parse_dates=['Date'])
combined.head()

## 2. Returns, volatility & moving averages (per ticker)

In [ ]:
per_ticker = {}
summaries = []

for ticker in TICKERS:
    df = combined[combined['Ticker'] == ticker].copy()
    df = an.add_returns(df)
    df = an.add_moving_averages(df)
    df = an.add_rolling_volatility(df)
    per_ticker[ticker] = df
    summaries.append(an.summarize_ticker(df, ticker))

summary_df = pd.DataFrame(summaries).set_index('Ticker')
summary_df

In [ ]:
# Price + moving averages chart for one ticker (change as needed)
# Note: SMA_200 only starts plotting once 200 real trading days exist —
# it's left as NaN before that (not a distorted "avg of 1 day"), so the
# line correctly starts partway through the chart.
ticker = 'AAPL'
df = per_ticker[ticker]

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Date'], y=df['Adj_Close'], name='Adj Close'))
fig.add_trace(go.Scatter(x=df['Date'], y=df['SMA_20'], name='SMA 20'))
fig.add_trace(go.Scatter(x=df['Date'], y=df['SMA_50'], name='SMA 50'))
fig.add_trace(go.Scatter(x=df['Date'], y=df['SMA_200'], name='SMA 200'))
fig.update_layout(title=f'{ticker} Price & Moving Averages', xaxis_title='Date', yaxis_title='Price')
fig.show()

In [ ]:
# Cumulative returns comparison across all tickers
fig = go.Figure()
for ticker, df in per_ticker.items():
    cum = an.cumulative_return(df)
    fig.add_trace(go.Scatter(x=df['Date'], y=cum, name=ticker))
fig.update_layout(title='Cumulative Returns (Growth of $1)', xaxis_title='Date', yaxis_title='Growth')
fig.show()

## 3. Risk metrics

In [ ]:
risk_rows = [rm.risk_summary(df, ticker) for ticker, df in per_ticker.items()]
risk_df = pd.DataFrame(risk_rows).set_index('Ticker')
risk_df

In [ ]:
# Drawdown chart — visualizes exactly where each ticker's Max_Drawdown
# figure above comes from, and how long it took to recover.
fig = go.Figure()
for ticker, df in per_ticker.items():
    dd = rm.drawdown_series(df)
    fig.add_trace(go.Scatter(x=df['Date'], y=dd, name=ticker))
fig.update_layout(title='Drawdown from Running Peak', xaxis_title='Date', yaxis_title='Drawdown',
                   yaxis_tickformat='.0%')
fig.show()

In [ ]:
# Beta relative to an equal-weight benchmark of all 5 tickers.
# (No external market index is pulled in this project, so the benchmark
# is this basket itself — beta here means "sensitivity to the basket",
# not literally to the S&P 500/Nifty.)
bench_weights = pf.equal_weight_portfolio(list(TICKERS.keys()))
returns_wide_all = pf.wide_returns(combined)
benchmark_returns = pf.portfolio_returns(returns_wide_all, bench_weights)

betas = {t: rm.beta(returns_wide_all[t], benchmark_returns) for t in TICKERS}
pd.Series(betas, name='Beta_vs_Basket').to_frame()

## 4. Portfolio comparison

In [ ]:
returns_wide = pf.wide_returns(combined)
corr = pf.correlation_matrix(returns_wide)

fig = px.imshow(corr, text_auto='.2f', title='Return Correlation Matrix', color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.show()

In [ ]:
# Compare an equal-weight portfolio against each individual stock
weights = pf.equal_weight_portfolio(list(TICKERS.keys()))
perf = pf.portfolio_performance(returns_wide, weights)
print('Equal-weight portfolio:', perf)

In [ ]:
# Efficient frontier via random simulation, with the two optimizer solutions
# (true min-volatility and true max-Sharpe, found via scipy.optimize.minimize
# with a budget constraint and no-shorting bounds) marked on top. The random
# cloud approximates the frontier; the optimizer solves for its exact edge.
sim = pf.random_portfolios(returns_wide, n_portfolios=3000)
min_vol = pf.optimize_portfolio(returns_wide, objective='min_vol')
max_sharpe = pf.optimize_portfolio(returns_wide, objective='max_sharpe')

fig = px.scatter(sim, x='Volatility', y='Return', color='Sharpe',
                  title='Simulated Portfolios (Efficient Frontier) + Optimizer Solutions',
                  color_continuous_scale='Viridis', opacity=0.5)
fig.add_trace(go.Scatter(x=[min_vol['Annualized_Volatility']], y=[min_vol['Annualized_Return']],
                          mode='markers', marker=dict(size=16, color='blue', symbol='star'),
                          name='Min Volatility (optimizer)'))
fig.add_trace(go.Scatter(x=[max_sharpe['Annualized_Volatility']], y=[max_sharpe['Annualized_Return']],
                          mode='markers', marker=dict(size=16, color='red', symbol='star'),
                          name='Max Sharpe (optimizer)'))
fig.show()

print('Min-volatility portfolio:', {k: v for k, v in min_vol.items() if k != 'weights'})
print('  weights:', {t: round(w, 3) for t, w in min_vol['weights'].items()})
print()
print('Max-Sharpe portfolio:', {k: v for k, v in max_sharpe.items() if k != 'weights'})
print('  weights:', {t: round(w, 3) for t, w in max_sharpe['weights'].items()})

## 5. Bonus: forecasting

In [ ]:
# Backtest first: hold out the last 30 trading days, forecast them with both
# methods trained on everything before, and score against what actually
# happened. This is what tells us whether the forecast below is trustworthy,
# rather than just plausible-looking.
ticker = 'AAPL'
df = per_ticker[ticker]

scores = fc.backtest(df, test_size=30)
print(f'{ticker} backtest (last 30 trading days held out):')
for k, v in scores.items():
    print(f'  {k}: {v}')

In [ ]:
# Now forecast forward for real, using an ARIMA order chosen by AIC search
# (select_arima_order) instead of a fixed, arbitrary (5,1,0).
linear_fc = fc.linear_trend_forecast(df, horizon=30)
arima_fc = fc.arima_forecast(df, horizon=30)
print(f'ARIMA order selected for {ticker}: {arima_fc.attrs["order"]}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Date'], y=df['Adj_Close'], name='Historical'))
fig.add_trace(go.Scatter(x=linear_fc['Date'], y=linear_fc['Forecast'], name='Linear Trend Forecast'))
fig.add_trace(go.Scatter(x=arima_fc['Date'], y=arima_fc['Forecast'], name='ARIMA Forecast'))
fig.add_trace(go.Scatter(x=arima_fc['Date'], y=arima_fc['Upper_CI'], line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=arima_fc['Date'], y=arima_fc['Lower_CI'], line=dict(width=0), fill='tonexty',
                          fillcolor='rgba(99,110,250,0.2)', name='ARIMA 95% CI'))
fig.update_layout(title=f'{ticker} 30-Day Forecast (backtested {scores["Better_Model"]} performed better historically)',
                   xaxis_title='Date', yaxis_title='Price')
fig.show()

## 6. Export summary tables for the dashboard

These CSVs feed directly into Power BI / Tableau.

In [ ]:
summary_df.to_csv('../data/processed/return_volatility_summary.csv')
risk_df.to_csv('../data/processed/risk_summary.csv')
corr.to_csv('../data/processed/correlation_matrix.csv')

optimal_weights_df = pd.DataFrame([
    {'Portfolio': 'Min_Volatility', **min_vol['weights']},
    {'Portfolio': 'Max_Sharpe', **max_sharpe['weights']},
    {'Portfolio': 'Equal_Weight', **weights},
]).set_index('Portfolio')
optimal_weights_df.to_csv('../data/processed/optimal_portfolio_weights.csv')

print('Exported summary tables to data/processed/')